<div style="padding: 10px; border-left: 6px solid #28a745; background-color: #d4edda; color: #155724;">
<strong>Success!</strong> Your model trained successfully.
</div>


<div style="padding: 10px; border-left: 6px solid #ffc107; background-color: #fff3cd; color: #856404;">
<strong>Warning!</strong> You have missing values in your dataset.
</div>


<div style="padding: 10px; border-left: 6px solid #17a2b8; background-color: #d1ecf1; color: #0c5460;">
<strong>Note:</strong> You can skip this step if using default settings.
</div>


<div style="padding: 10px; border-left: 6px solid #dc3545; background-color: #f8d7da; color: #721c24;">
<strong>Error!</strong> Failed to load the data file.
</div>


## 01- Model

<div style="text-align: center;">
  <img src="./transformer_1.png" alt="Sample Image" width="300" height="400"/>
</div>


In [1]:
import torch
import torch.nn as nn
import math

#### Input Embedding

In [ ]:
class InputEmbedding(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__() # Initialize the parent class
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model) # This creates a matrix of shape (vocab_size, d_model)

    def forward(self, x):
        #  In the embedding layers, we multiply those weights by sqrt(d_model) 
        # This was mentioned in the attention research paper
        return self.embedding(x) * math.sqrt(self.d_model)

#### Positional Encoding

$$
\begin{aligned}
\text{For even positions: } &\quad \text{PE}(pos, 2i) = \sin\left( \frac{pos}{10000^{\frac{2i}{d_{\text{model}}}}} \right) \\
\text{For odd positions: } &\quad \text{PE}(pos, 2i+1) = \cos\left( \frac{pos}{10000^{\frac{2i}{d_{\text{model}}}}} \right)
\end{aligned}
$$

$$
10000^{\frac{2i}{d_{\text{model}}}} = \exp\left( \frac{2i}{d_{\text{model}}} \cdot \log(10000) \right)
$$


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model:int, seq_len:int, dropout:float) -> None:
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)
        
        # creting a tensor of zeros with shape (seq_len, d_model)
        pe = torch.zeros(seq_len, d_model)
        # creating a tensor of positions with shape (seq_len, 1)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
        # this is the dinominator for the positional encoding formula
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        # Even positions - Apply sine function
        pe[:, 0::2] = torch.sin(position * div_term)
        # Odd positions - Apply cosine function
        pe[:, 1::2] = torch.cos(position * div_term)

        # Adding a new dimension to the tensor for batch processing
        pe = pe.unsqueeze(0)
        # Registering the positional encoding as a buffer so it is not a model parameter
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + (self.pe[:, :x.size(1)]).requires_grad_(False)
        return self.dropout(x)

#### Layer Normalization

#### 📏 Layer Normalization in Transformers

**Layer Normalization** is used in Transformers to stabilize and accelerate training by normalizing the inputs across the **feature dimension** (not across the batch like BatchNorm).
#### 🔹 Formula:
$$
\text{LayerNorm}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta
$$
Where:
- $\mu$: Mean of the feature vector
- $\sigma^2$: Variance of the feature vector
- $\epsilon$: Small constant to prevent division by zero
- $\gamma$, $\beta$: Learnable scale and shift parameters


In [ ]:
class LayerNormalization(nn.Module):
    # epsilon is a small value and is used for numerical stability
    # It prevents division by zero, and also cpu/gpy can only represent a limited range of numbers
    # so we need to add a small value to the denominator to avoid overflow
    def __init__(self, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        ## nn.paramter = This makes the parameters learnable
        self.alpha = nn.Parameter(torch.ones(1))  # Scale parameter (Multiplied)
        self.bias = nn.Parameter(torch.zeros(1))  # Shift parameter (Added)
    
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        return self.alpha * (x - mean) / (std + self.eps) + self.bias

#### Feed Forward Network (FFN)

#### 🔥 Position-wise Feed-Forward Networks (FFN)

In the Transformer, each encoder and decoder layer contains a **feed-forward network** that is applied **independently** to each token position.

#### 🔹 Formula

$$
\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2
$$

- Two linear layers with a ReLU in between
- First layer projects up to a higher dimension (`d_ff`)
- Second layer projects back to original size (`d_model`)



In [ ]:
class FeedForwardNetwork(nn.module):
    def __init__(self, d_model:int, d_ff:int, dropout:float) -> None:
        super().__init__()
        # nn.linear -> y = xW + b
        # where W is the weight matrix and b is the bias vector
        self.linear1 = nn.Linear(d_model, d_ff)  # First linear layer -> W1 and b1
        self.dropout = nn.Dropout(dropout)  # Dropout layer
        self.linear2 = nn.Linear(d_ff, d_model)  # Second linear layer -> W2 and b2

    def forward(self, x):
        # (Batch, Seq_Len, d_model) --> (Batch, Seq_Len, d_ff) --> (Batch, Seq_Len, d_model)
        return self.linear2(self.dropout(torch.relu(self.linear1(x))))

###  Multi-Head Attention

Multi-head attention is a key mechanism in the Transformer that allows the model to focus on different parts of the input sequence **simultaneously**.

---

#### 🔹 Why Multi-Head?
- A single attention head might focus on only one type of relationship (e.g., word similarity).
- Multi-head attention allows the model to learn **diverse relationships** from multiple representation subspaces.
- Each head can attend to different parts or patterns in the sequence.

---

#### 🔹 How it Works (Conceptually):

1. **Input Projections**:
   - Each head gets its own set of learned linear projections: \( W_i^Q, W_i^K, W_i^V \)
   - These reduce the dimensionality from \( d_{model} \) to smaller \( d_k \) and \( d_v \)

2. **Scaled Dot-Product Attention**:
   - Attention is calculated independently in each head:
   $$
   \text{Attention}(Q, K, V) = \text{softmax}\left( \frac{QK^T}{\sqrt{d_k}} \right) V
   $$

3. **Concatenate Outputs**:
   - The outputs of all heads are concatenated together: 
   $$
   \text{Concat}(\text{head}_1, \dots, \text{head}_h)
   $$

4. **Final Linear Projection**:
   - The concatenated output is projected back to \( d_{model} \) using a learned matrix \( W^O \)

---

#### 🔹 Benefits:
- Allows the model to **attend to multiple features or positions** at once
- Enables learning of **richer, multi-faceted relationships** between tokens
- **Efficient**, since each head operates on lower dimensions

---

#### 🔹 Common Setup:
- \( d_{model} = 512 \)
- \( h = 8 \) heads
- \( d_k = d_v = 64 \) for each head
- Total concatenated size = \( 8 \times 64 = 512 \)



<div style="text-align: center;">
  <img src="./multihead_attention.png" alt="Sample Image" width="500" height="300"/>
</div>


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model:int, h:int, dropout:float) -> None:
        super().__init__()
        self.d_model = d_model
        self.h = h
        # make sure d_model is divisible by h
        assert d_model % h == 0, "d_model must be divisible by h"
        self.d_k = d_model // h

        # nn.linear -> y = xW + b
        # where W is the weight matrix and b is the bias vector
        self.w_q = nn.Linear(d_model, d_model)  # Query linear transformation
        self.w_k = nn.Linear(d_model, d_model)  # Key linear transformation
        self.w_v = nn.Linear(d_model, d_model) # Value linear transformation
        self.w_o = nn.Linear(d_model, d_model) # Output linear transformation

        self.dropout = nn.Dropout(dropout)  # Dropout layer
    
    @staticmethod
    def attention(self, query, key, value, mask, dropout:nn.Dropout):
        d_k = query.shape[-1]  # Get the last dimension of the query tensor
        # (Batch, h, Seq_Len, d_k) @ (Batch, h, d_k, Seq_Len) --> (Batch, h, Seq_Len, Seq_Len)
        # For each batch and each head, (Seq_Len, d_k) @ (d_k, Seq_Len) → (Seq_Len, Seq_Len)
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)  # Scaled dot-product attention

        # before applying softmax, we need to apply the mask
        if mask is not None:
            attention_scores.masked_fill_(mask == 0, float('-inf'))  # Set masked positions to -inf
        
        attention_scores = attention_scores.softmax(dim=-1) # (Batch, h, Seq_Len, Seq_Len)

        # Apply dropout to the attention scores
        if dropout is not None:
            attention_scores = dropout(attention_scores)
        
        # attention scores are returned for visualization purposes
        return (attention_scores @ value), attention_scores  # (Batch, h, Seq_Len, d_k), attention_scores


    def forward(self, q, k, v, mask):
        query = self.w_q(q)  # (Batch, Seq_Len, d_model) --> (Batch, Seq_Len, d_model)
        key = self.w_k(k)    # (Batch, Seq_Len, d_model) --> (Batch, Seq_Len, d_model)
        value = self.w_v(v)  # (Batch, Seq_Len, d_model) --> (Batch, Seq_Len, d_model)

        # dividing query, key, value into h heads
        # view() is used to keep batch dimension cause we dont want to split the sentence
        # we also dont need to split sequence length dimension
        # we just want to split the embedding dimension
        # (Batch, Seq_Len, d_model) --> (Batch, Seq_Len, h, d_k) --> (Batch, h, seq_Len, d_k)
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1, 2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1, 2)

        x, self.attention_scores = MultiHeadAttention.attention(query, key, value, mask, self.dropout)

        # (Batch, h, Seq_Len, d_k) --> (Batch, Seq_Len, h, d_k) --> (Batch, Seq_Len, d_model)
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.h * self.d_k)

        # (Batch, Seq_Len, d_model) --> (Batch, Seq_Len, d_model) 
        return self.w_o(x)


